In [7]:
import os
import json
import csv
from bs4 import BeautifulSoup
import re

def parse_avito_from_path(file_path):
    """Парсит страницу Avito"""
    
    if not os.path.exists(file_path):
        print(f"Файл {file_path} не найден!")
        return []
    
    try:
        encodings = ['utf-8', 'cp1251', 'windows-1251']
        html_content = None
        
        for encoding in encodings:
            try:
                with open(file_path, 'r', encoding=encoding) as f:
                    html_content = f.read()
                print(f"Файл успешно прочитан с кодировкой: {encoding}")
                break
            except UnicodeDecodeError:
                continue
        
        if html_content is None:
            print("Не удалось прочитать файл.")
            return []
            
    except Exception as e:
        print(f"Ошибка чтения файла: {e}")
        return []
    
    soup = BeautifulSoup(html_content, 'html.parser')
    cars_data = []
    
    print("Поиск объявлений...")
    
    selectors = [
        'div[data-marker="item"]',
        '.iva-item-root-_lk9K',
        '.items-items-kAJAg .iva-item-root-_lk9K',
        '[data-marker="catalog-serp"] [data-marker="item"]',
        '.iva-item-body-KLUuy'
    ]
    
    car_items = []
    for selector in selectors:
        found = soup.select(selector)
        if found:
            print(f"Найдено {len(found)} элементов с селектором: {selector}")
            car_items = found
            break
    
    if not car_items:
        print("Не найдены объявления.")
        
        print("\n=== ОТЛАДОЧНАЯ ИНФОРМАЦИЯ ===")
        
        if "avito" in html_content.lower():
            print("✓ Текст 'avito' найден в странице")
        else:
            print("✗ Текст 'avito' не найден - возможно файл поврежден")
        
        title = soup.find('title')
        if title:
            print(f"Заголовок страницы: {title.get_text()}")
        
        car_keywords = ['авто', 'машина', 'bmw', 'audi', 'mercedes', 'toyota', 'honda']
        found_keywords = []
        for keyword in car_keywords:
            if keyword in html_content.lower():
                found_keywords.append(keyword)
        
        if found_keywords:
            print(f"Найдены ключевые слова: {found_keywords}")
        else:
            print("Ключевые слова автомобилей не найдены")
        
        return []
    
    print(f"Обрабатывается {len(car_items)} объявлений...")
    
    for i, item in enumerate(car_items):
        car_data = {}
        
        car_data['id'] = item.get('data-item-id', f'item_{i}')
        
        title_selectors = [
            'a[data-marker="item-title"]',
            '.iva-item-titleStep-pdebR a',
            'h3[itemprop="name"] a',
            '[data-marker="item-title"]'
        ]
        
        for selector in title_selectors:
            title_elem = item.select_one(selector)
            if title_elem:
                car_data['title'] = title_elem.get_text(strip=True)
                href = title_elem.get('href', '')
                if href:
                    car_data['url'] = f"https://www.avito.ru{href}" if href.startswith('/') else href
                break
        
        price_selectors = [
            '[data-marker="item-price"]',
            '.iva-item-priceStep-UqBUr',
            'meta[itemprop="price"]',
            '[itemprop="price"]'
        ]
        
        for selector in price_selectors:
            price_elem = item.select_one(selector)
            if price_elem:
                price_text = price_elem.get_text(strip=True)
                car_data['price'] = price_text
                # Извлекаем цифры из цены
                numbers = re.findall(r'[\d\s]+', price_text)
                if numbers:
                    car_data['price_number'] = numbers[0].replace(' ', '')
                break
        
        param_selectors = [
            '[data-marker="item-specific-params"]',
            '.iva-item-autoParamsStep-WzfS8',
            '[itemprop="description"]'
        ]
        
        for selector in param_selectors:
            param_elem = item.select_one(selector)
            if param_elem:
                params_text = param_elem.get_text(strip=True)
                car_data['params'] = params_text
                
                parts = [p.strip() for p in params_text.split(',')]
                for part in parts:
                    if any(c.isdigit() for c in part):
                        if 'км' in part:
                            car_data['mileage'] = part
                        elif 'л' in part:
                            car_data['engine'] = part
                        elif len(part) == 4 and part.isdigit():
                            car_data['year'] = part
                break
        
        location_selectors = [
            '[data-marker="item-location"]',
            '.iva-item-locationStep-3BjUJ',
            '[data-marker="item-address"]'
        ]
        
        for selector in location_selectors:
            loc_elem = item.select_one(selector)
            if loc_elem:
                car_data['location'] = loc_elem.get_text(strip=True)
                break
        
        date_selectors = [
            '[data-marker="item-date"]',
            '.iva-item-dateInfoStep-_acjp'
        ]
        
        for selector in date_selectors:
            date_elem = item.select_one(selector)
            if date_elem:
                car_data['date'] = date_elem.get_text(strip=True)
                break
        
        desc_selectors = [
            '.iva-item-descriptionStep-C0ty1',
            '[data-marker="item-specific-params"]'
        ]
        
        for selector in desc_selectors:
            desc_elem = item.select_one(selector)
            if desc_elem:
                car_data['description'] = desc_elem.get_text(strip=True)
                break
        
        cars_data.append(car_data)
    
    return cars_data

def save_cars_data(cars_data, prefix='avito_cars'):
    """Сохраняет данные в JSON и CSV"""
    
    if not cars_data:
        print("Нет данных для сохранения")
        return
    
    json_file = f'{prefix}.json'
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(cars_data, f, ensure_ascii=False, indent=2)
    
    csv_file = f'{prefix}.csv'
    with open(csv_file, 'w', encoding='utf-8', newline='') as f:
        if cars_data:
            fieldnames = set()
            for car in cars_data:
                fieldnames.update(car.keys())
            fieldnames = list(fieldnames)
            
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for car in cars_data:
                row = {field: car.get(field, '') for field in fieldnames}
                writer.writerow(row)
    
    print(f"Данные сохранены в {json_file} и {csv_file}")
    return json_file, csv_file

def main():
    print("=" * 60)
    print("ПАРСЕР AVITO - РЕЖИМ ЧТЕНИЯ ИЗ ФАЙЛА")
    print("=" * 60)
    
    file_path = r"C:\Users\Мартинсон Диана\Downloads\avito_cars.html"
    
    print(f"Используется файл: {file_path}")
    print()
    
    cars_data = parse_avito_from_path(file_path)
    
    if not cars_data:
        print("\nНе удалось извлечь данные из файла.")
        return
    
    json_file, csv_file = save_cars_data(cars_data)
    
    print(f"\n✅ УСПЕХ! Обработано {len(cars_data)} объявлений")
    
    print(f"\nПЕРВЫЕ 5 ОБЪЯВЛЕНИЙ:")
    for i, car in enumerate(cars_data[:5], 1):
        print(f"\n{i}. {car.get('title', 'Без названия')}")
        print(f"   Цена: {car.get('price', 'Не указана')}")
        if 'params' in car:
            print(f"   Параметры: {car['params']}")
        if 'location' in car:
            print(f"   Место: {car['location']}")
        if 'date' in car:
            print(f"   Дата: {car['date']}")
    
    prices = []
    for car in cars_data:
        if 'price_number' in car and car['price_number']:
            try:
                price = int(car['price_number'])
                prices.append(price)
            except:
                pass
    
    if prices:
        print(f"\nСТАТИСТИКА ПО ЦЕНАМ:")
        print(f"   Минимальная цена: {min(prices):,} руб.")
        print(f"   Максимальная цена: {max(prices):,} руб.")
        print(f"   Средняя цена: {sum(prices)/len(prices):,.0f} руб.")

if __name__ == "__main__":
    main()

ПАРСЕР AVITO - РЕЖИМ ЧТЕНИЯ ИЗ ФАЙЛА
Используется файл: C:\Users\Мартинсон Диана\Downloads\avito_cars.html

Файл успешно прочитан с кодировкой: utf-8
Поиск объявлений...
Найдено 50 элементов с селектором: div[data-marker="item"]
Обрабатывается 50 объявлений...
Данные сохранены в avito_cars.json и avito_cars.csv

✅ УСПЕХ! Обработано 50 объявлений

ПЕРВЫЕ 5 ОБЪЯВЛЕНИЙ:

1. ВАЗ (LADA) Vesta 1.6 MT, 2019, 145 000 км
   Цена: 599 000₽
   Параметры: 145 000 км, 1.6 MT (106 л.с.), седан, передний, бензин
   Место: Краснодарский край, Курганинский р-н, Курганинское городское поселение, Курганинск

2. ВАЗ (LADA) Priora 1.6 MT, 2018, 181 000 км
   Цена: 750 000₽
   Параметры: 181 000 км, 1.6 MT (106 л.с.), седан, передний, бензин
   Место: Краснодарский край, Краснодар

3. Opel Astra 1.6 AT, 2014, 67 000 км
   Цена: 915 000₽
   Параметры: 67 000 км, 1.6 AT (115 л.с.), хетчбэк, передний, бензин
   Место: Тульская обл., Тула

4. Mazda 3 1.6 AT, 2008, 208 500 км
   Цена: 549 000₽
   Параметры: 208 